In [3]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from utils import ALL_LETTERS ,N_LETTERS
from utils import load_data , letter_to_tensor , line_to_tensor , random_training_example


class RNN(nn.Module):
  def __init__(self,input_size,hidden_size,output_size):
    super(RNN,self).__init__()

    self.hidden_size = hidden_size
    self.i2h = nn.Linear(input_size + hidden_size, hidden_size) #input is previous hidden and input and the output is hidden
    self.i2o = nn.Linear(input_size + hidden_size, output_size)
    self.softmax = nn.LogSoftmax(dim=1) #input is (1,57)


  def forward (self , input_tensor , hidden_tensor):
    combined = torch.cat((input_tensor , hidden_tensor) , 1)
    hidden = self.i2h(combined)
    output = self.i2o(combined)
    output = self.softmax(output)
    return output , hidden



  def init_hidden(self):
    return torch.zeros(1,self.hidden_size)

n_hidden = 128

category_lines , all_categories = load_data()
n_categories = len(all_categories)
print(n_categories)

rnn=RNN(N_LETTERS,n_hidden,n_categories)
n_hidden = 128

#one step
input_tensor = letter_to_tensor('A')
hidden_tensor = rnn.init_hidden()
output , next_hidden = rnn(input_tensor , hidden_tensor)
print(output.size())
print(next_hidden.size())
print(output)



# whole sequence/name
input_tensor = line_to_tensor('Albert')
hidden_tensor = rnn.init_hidden()

output, next_hidden = rnn(input_tensor[0], hidden_tensor)
#print(output.size())
#print(next_hidden.size())



def category_from_output(output):
    category_idx = torch.argmax(output).item()
    return all_categories[category_idx]

print(category_from_output(output))



criterion = nn.NLLLoss()
learning_rate = 0.005
optimizer = torch.optim.SGD(rnn.parameters(), lr=learning_rate)

def train(line_tensor, category_tensor):
    hidden = rnn.init_hidden()

    for i in range(line_tensor.size()[0]):
        output, hidden = rnn(line_tensor[i], hidden)

    loss = criterion(output, category_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return output, loss.item()

current_loss = 0
all_losses = []
plot_steps, print_steps = 1000, 5000
n_iters = 150000
for i in range(n_iters):
    category, line, category_tensor, line_tensor = random_training_example(category_lines, all_categories)

    output, loss = train(line_tensor, category_tensor)
    current_loss += loss

    if (i+1) % plot_steps == 0:
        all_losses.append(current_loss / plot_steps)
        current_loss = 0

    if (i+1) % print_steps == 0:
        guess = category_from_output(output)
        correct = "CORRECT" if guess == category else f"WRONG ({category})"
        print(f"{i+1} {(i+1)/n_iters*100} {loss:.4f} {line} / {guess} {correct}")




def predict(input_line):
    print(f"\n> {input_line}")
    with torch.no_grad():
        line_tensor = line_to_tensor(input_line)

        hidden = rnn.init_hidden()

        for i in range(line_tensor.size()[0]):
            output, hidden = rnn(line_tensor[i], hidden)

        guess = category_from_output(output)
        print(guess)


while True:
    sentence = input("Input:")
    if sentence == "quit":
        break

    predict(sentence)


18
torch.Size([1, 18])
torch.Size([1, 128])
tensor([[-2.7869, -2.9459, -2.9316, -2.9403, -2.9587, -2.9568, -2.8394, -2.9400,
         -2.8122, -2.9726, -2.8716, -2.9568, -2.9857, -2.8958, -2.8547, -2.7914,
         -2.8099, -2.8174]], grad_fn=<LogSoftmaxBackward0>)
English
5000 3.3333333333333335 2.5648 Wyrick / Czech WRONG (Polish)
10000 6.666666666666667 2.1207 Ferreira / Spanish WRONG (Portuguese)
15000 10.0 2.0172 Achthoven / Dutch CORRECT
20000 13.333333333333334 1.1230 Berger / German CORRECT
25000 16.666666666666664 3.8051 Reynold / Scottish WRONG (Irish)
30000 20.0 2.2031 Araya / Japanese WRONG (Spanish)
35000 23.333333333333332 5.3080 Plamondon / Scottish WRONG (French)
40000 26.666666666666668 2.2630 Kunze / English WRONG (German)
45000 30.0 0.2475 Chellos / Greek CORRECT
50000 33.33333333333333 1.1885 Koury / Arabic CORRECT
55000 36.666666666666664 0.8564 Chi / Korean CORRECT
60000 40.0 0.4001 Schoettmer / German CORRECT
65000 43.333333333333336 0.9736 Mifsud / Arabic CORREC

KeyboardInterrupt: Interrupted by user